# RainGuard — Module 6: Rainfall Intensity Classification

**Indian Rainfall Forecasting and Heavy Rainfall Risk Analysis System**

## Objective
Predict the **next calendar day's rainfall intensity** for each district using only information available at the end of the current day.

This notebook follows Module 4's leakage-safe targets and feature groups and uses the **same chronological split boundary saved by Module 5**.

### Classification tasks
1. **Multiclass intensity classification** — `target_next_class` (the IMD intensity class assigned in Module 4 using the saved thresholds).
2. **Heavy-or-above classification** — `target_next_heavy` (1 when next-day rainfall is at least the Module 4 `heavy_threshold_mm`, otherwise 0). This second task is a focused operational warning experiment.

### Models
- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier
- Gradient Boosting Classifier

### Evaluation
Accuracy, balanced accuracy, macro precision/recall/F1, weighted F1, per-class precision/recall/F1, confusion matrices, and recall for the heavy-or-above class.

**Important:** This dataset covers a short date window. Results are a short-horizon demonstration, not a year-round or flood-occurrence forecast. Heavy-rainfall classification is not the same as predicting actual floods.


## Step 1 — Imports, project paths and saved configuration

Run this notebook from `RainGuard/notebooks/`. If Jupyter starts elsewhere, update `PROJECT_ROOT` manually in the path cell.

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(cwd, "..")) if os.path.basename(cwd) == "notebooks" else cwd
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "rainfall_features.csv")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
FIG_DIR = os.path.join(PROJECT_ROOT, "docs", "figures")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

CONFIG_PATH = os.path.join(MODEL_DIR, "feature_config.json")
assert os.path.exists(DATA_PATH), f"Feature dataset not found: {DATA_PATH}. Run Module 4 first."
assert os.path.exists(CONFIG_PATH), f"feature_config.json not found: {CONFIG_PATH}. Run Module 4 first."

with open(CONFIG_PATH, encoding="utf-8") as f:
    config = json.load(f)

print("Project root:", PROJECT_ROOT)
print("Feature dataset:", DATA_PATH)
print("Model directory:", MODEL_DIR)
print("Figure directory:", FIG_DIR)


## Step 2 — Load and validate the engineered dataset

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["date", "target_date"])

TARGET_MULTI = config["targets"]["classification"]
TARGET_BINARY = config["targets"]["heavy_binary"]
FEATURE_GROUPS = config["feature_groups"]
INTENSITY_LABELS = config["intensity_labels"]
HEAVY_MM = float(config["heavy_threshold_mm"])

print("Shape:", df.shape)
print("Districts:", df["district"].nunique())
print("Forecast dates:", df["target_date"].min().date(), "to", df["target_date"].max().date())
print("Multiclass target:", TARGET_MULTI)
print("Binary target:", TARGET_BINARY)
print("Intensity labels:", INTENSITY_LABELS)
print("Heavy threshold (mm):", HEAVY_MM)

display(df.head())


In [ ]:
required = [TARGET_MULTI, TARGET_BINARY, "target_next_mm", "date", "target_date"]
required += [c for group in FEATURE_GROUPS.values() for c in group]
missing_cols = sorted(set(required) - set(df.columns))
assert not missing_cols, f"Missing required columns: {missing_cols}"
assert df[required].notna().all().all(), "Missing values found in required targets/features."
assert set(df[TARGET_BINARY].unique()).issubset({0, 1}), "Binary target must contain only 0 and 1."
assert (df["target_date"] - df["date"]).dt.days.eq(1).all(), "Target date is not exactly one day after feature date."
assert not df.duplicated(["state", "district", "date"]).any(), "Duplicate district-date rows found."

print("Validation passed.")
print("\nMulticlass distribution:")
display(df[TARGET_MULTI].value_counts().reindex(INTENSITY_LABELS, fill_value=0).rename("rows").to_frame())
print("\nHeavy-or-above distribution:")
display(df[TARGET_BINARY].value_counts().rename(index={0:"Below heavy threshold",1:"Heavy or above"}).to_frame("rows"))


## Step 3 — Define feature sets

These feature groups are read from Module 4's `feature_config.json`; do not manually add forbidden weekly, monthly, cumulative, departure or same-day category columns.

In [ ]:
history = FEATURE_GROUPS["history"]
season = FEATURE_GROUPS["season"]
location_numeric = FEATURE_GROUPS["location_numeric"]
location_codes = FEATURE_GROUPS["location_codes"]
climatology = FEATURE_GROUPS["climatology"]

FEATURE_SETS = {
    "History only": history,
    "History + season + district mean": history + season + location_numeric,
    "History + season + district mean + climatology": history + season + location_numeric + climatology,
    "Expanded (including location codes)": history + season + location_numeric + climatology + location_codes,
}

for name, cols in FEATURE_SETS.items():
    assert len(cols) == len(set(cols)), f"Duplicate features in {name}"
    assert all(c in df.columns for c in cols), f"Missing features in {name}"
    print(f"{name}: {len(cols)} features")
    print(cols)


## Step 4 — Reuse Module 5's chronological train/test boundary

The split is based on forecast date, not random rows. All districts on a given forecast date stay in the same partition. This prevents later dates from entering training when earlier dates are being evaluated.

In [ ]:
assert "regression_split_date" in config, (
    "No regression_split_date found. Run Module 5 through its chronological split step first."
)
split_date = pd.Timestamp(config["regression_split_date"])

train_df = df[df["target_date"] < split_date].copy()
test_df = df[df["target_date"] >= split_date].copy()

print("Shared split date:", split_date.date())
print("Train forecast dates:", train_df["target_date"].min().date(), "to", train_df["target_date"].max().date())
print("Test forecast dates:", test_df["target_date"].min().date(), "to", test_df["target_date"].max().date())
print("Train rows:", len(train_df), "| Test rows:", len(test_df))

assert len(train_df) > 0 and len(test_df) > 0
assert train_df["target_date"].max() < test_df["target_date"].min()


## Step 5 — Check class imbalance

Rainfall classes are usually imbalanced: no/light rainfall can dominate while very heavy classes are rare. Report class counts and do not interpret accuracy alone.

In [ ]:
class_counts = (
    train_df[TARGET_MULTI].value_counts()
    .reindex(INTENSITY_LABELS, fill_value=0)
    .rename("training_rows")
    .to_frame()
)
class_counts["training_percent"] = (class_counts["training_rows"] / len(train_df) * 100).round(2)
display(class_counts)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=class_counts.index, y=class_counts["training_rows"], ax=ax)
ax.set_title("Training Set: Next-Day Rainfall Intensity Distribution")
ax.set_xlabel("Rainfall intensity class")
ax.set_ylabel("Number of observations")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m6_01_class_distribution.png"), dpi=160, bbox_inches="tight")
plt.show()

print("Heavy-or-above training share:", round(train_df[TARGET_BINARY].mean() * 100, 2), "%")


## Step 6 — Train and compare multiclass classifiers

All models use the same chronological split and each feature set. Imputation and scaling (where needed) are fitted inside a pipeline using training data only. Tree models do not require scaling, but using a pipeline keeps preprocessing reproducible.

The primary selection metric is **macro F1**, which gives each class equal weight. Also inspect heavy-class recall and the confusion matrix.

In [ ]:
MULTI_MODELS = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(
            max_depth=12, min_samples_leaf=10, class_weight="balanced", random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=250, min_samples_leaf=3, class_weight="balanced_subsample",
            n_jobs=-1, random_state=RANDOM_STATE
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
    ])
}

multi_results = []
multi_fitted = {}
multi_predictions = {}

for feature_set_name, features in FEATURE_SETS.items():
    X_train = train_df[features]
    X_test = test_df[features]
    y_train = train_df[TARGET_MULTI].astype(str)
    y_test = test_df[TARGET_MULTI].astype(str)

    for model_name, estimator in MULTI_MODELS.items():
        model = estimator
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        row = {
            "feature_set": feature_set_name,
            "model": model_name,
            "accuracy": accuracy_score(y_test, pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, pred),
            "macro_precision": precision_score(y_test, pred, average="macro", zero_division=0),
            "macro_recall": recall_score(y_test, pred, average="macro", zero_division=0),
            "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_test, pred, average="weighted", zero_division=0),
        }
        multi_results.append(row)
        multi_fitted[(feature_set_name, model_name)] = model
        multi_predictions[(feature_set_name, model_name)] = pred

multi_results_df = pd.DataFrame(multi_results).sort_values(
    ["macro_f1", "balanced_accuracy"], ascending=False
).reset_index(drop=True)

display(multi_results_df.round(4))


## Step 7 — Select the best multiclass model and inspect per-class performance

In [ ]:
best_multi_row = multi_results_df.iloc[0]
best_multi_key = (best_multi_row["feature_set"], best_multi_row["model"])
best_multi_model = multi_fitted[best_multi_key]
best_multi_pred = multi_predictions[best_multi_key]
best_multi_features = FEATURE_SETS[best_multi_key[0]]
y_test_multi = test_df[TARGET_MULTI].astype(str)

print("Best feature set:", best_multi_key[0])
print("Best classifier:", best_multi_key[1])
print("\nClassification report:")
print(classification_report(
    y_test_multi, best_multi_pred, labels=INTENSITY_LABELS,
    zero_division=0, digits=3
))

cm = confusion_matrix(y_test_multi, best_multi_pred, labels=INTENSITY_LABELS)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=INTENSITY_LABELS, yticklabels=INTENSITY_LABELS, ax=ax
)
ax.set_title("Best Multiclass Model — Confusion Matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m6_02_multiclass_confusion_matrix.png"), dpi=160, bbox_inches="tight")
plt.show()


## Step 8 — Heavy-or-above binary classification

This second classification task uses Module 4's `target_next_heavy` label. It is a rainfall-threshold warning experiment, **not a verified flood prediction model**. Recall is especially important because missed heavy-rainfall events are operationally costly, but precision must also be considered because false alarms matter.

In [ ]:
BINARY_MODELS = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(
            max_depth=10, min_samples_leaf=10, class_weight="balanced", random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=250, min_samples_leaf=3, class_weight="balanced_subsample",
            n_jobs=-1, random_state=RANDOM_STATE
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
    ])
}

binary_results = []
binary_fitted = {}
binary_predictions = {}

for feature_set_name, features in FEATURE_SETS.items():
    X_train = train_df[features]
    X_test = test_df[features]
    y_train = train_df[TARGET_BINARY].astype(int)
    y_test = test_df[TARGET_BINARY].astype(int)

    for model_name, estimator in BINARY_MODELS.items():
        model = estimator
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        binary_results.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "accuracy": accuracy_score(y_test, pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, pred),
            "precision_heavy": precision_score(y_test, pred, zero_division=0),
            "recall_heavy": recall_score(y_test, pred, zero_division=0),
            "f1_heavy": f1_score(y_test, pred, zero_division=0),
            "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
        })
        binary_fitted[(feature_set_name, model_name)] = model
        binary_predictions[(feature_set_name, model_name)] = pred

binary_results_df = pd.DataFrame(binary_results).sort_values(
    ["recall_heavy", "f1_heavy", "balanced_accuracy"], ascending=False
).reset_index(drop=True)

display(binary_results_df.round(4))


## Step 9 — Evaluate the best heavy-rainfall classifier

In [ ]:
best_binary_row = binary_results_df.iloc[0]
best_binary_key = (best_binary_row["feature_set"], best_binary_row["model"])
best_binary_model = binary_fitted[best_binary_key]
best_binary_pred = binary_predictions[best_binary_key]
best_binary_features = FEATURE_SETS[best_binary_key[0]]
y_test_binary = test_df[TARGET_BINARY].astype(int)

print("Selected using highest heavy-class recall, then F1 and balanced accuracy.")
print("Best feature set:", best_binary_key[0])
print("Best classifier:", best_binary_key[1])
print("\nClassification report:")
print(classification_report(
    y_test_binary, best_binary_pred,
    labels=[0, 1], target_names=["Below heavy threshold", "Heavy or above"],
    zero_division=0, digits=3
))

cm_binary = confusion_matrix(y_test_binary, best_binary_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_binary, annot=True, fmt="d", cmap="Oranges",
    xticklabels=["Below threshold", "Heavy or above"],
    yticklabels=["Below threshold", "Heavy or above"], ax=ax
)
ax.set_title("Heavy-or-Above Classifier — Confusion Matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m6_03_heavy_binary_confusion_matrix.png"), dpi=160, bbox_inches="tight")
plt.show()


## Step 10 — Compare classifier performance visually

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

multi_plot = multi_results_df.copy()
multi_plot["configuration"] = multi_plot["feature_set"] + " | " + multi_plot["model"]
sns.barplot(data=multi_plot, y="configuration", x="macro_f1", ax=axes[0])
axes[0].set_title("Multiclass: Macro F1 by Model and Feature Set")
axes[0].set_xlabel("Macro F1")
axes[0].set_ylabel("")

binary_plot = binary_results_df.copy()
binary_plot["configuration"] = binary_plot["feature_set"] + " | " + binary_plot["model"]
sns.barplot(data=binary_plot, y="configuration", x="recall_heavy", ax=axes[1])
axes[1].set_title("Heavy-or-Above: Recall by Model and Feature Set")
axes[1].set_xlabel("Recall for heavy-or-above class")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m6_04_classifier_comparison.png"), dpi=160, bbox_inches="tight")
plt.show()


## Step 11 — Save models, metrics, predictions and metadata

The saved model files are intended for Module 7/8 and the later prediction application. Keep the exact feature lists with each model.

In [ ]:
MULTI_MODEL_PATH = os.path.join(MODEL_DIR, "rainfall_intensity_classifier.pkl")
BINARY_MODEL_PATH = os.path.join(MODEL_DIR, "heavy_rainfall_classifier.pkl")
MULTI_METRICS_PATH = os.path.join(MODEL_DIR, "classification_multiclass_metrics.csv")
BINARY_METRICS_PATH = os.path.join(MODEL_DIR, "classification_heavy_metrics.csv")
MULTI_PRED_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "classification_multiclass_test_predictions.csv")
BINARY_PRED_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "classification_heavy_test_predictions.csv")

joblib.dump(best_multi_model, MULTI_MODEL_PATH)
joblib.dump(best_binary_model, BINARY_MODEL_PATH)
multi_results_df.to_csv(MULTI_METRICS_PATH, index=False)
binary_results_df.to_csv(BINARY_METRICS_PATH, index=False)

multi_diag = test_df[["state", "district", "date", "target_date", TARGET_MULTI]].copy()
multi_diag["predicted_class"] = best_multi_pred
multi_diag.to_csv(MULTI_PRED_PATH, index=False)

binary_diag = test_df[["state", "district", "date", "target_date", TARGET_BINARY, "target_next_mm"]].copy()
binary_diag["predicted_heavy"] = best_binary_pred
binary_diag.to_csv(BINARY_PRED_PATH, index=False)

metadata = {
    "project": "RainGuard",
    "task": "next-calendar-day district rainfall intensity classification",
    "split_date": str(split_date.date()),
    "multiclass": {
        "target": TARGET_MULTI,
        "labels": INTENSITY_LABELS,
        "best_model": best_multi_key[1],
        "feature_set": best_multi_key[0],
        "features": best_multi_features,
        "selection_metric": "macro_f1",
        "metrics": {k: float(best_multi_row[k]) for k in
                    ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]}
    },
    "heavy_binary": {
        "target": TARGET_BINARY,
        "positive_class": "Heavy or above",
        "threshold_mm": HEAVY_MM,
        "best_model": best_binary_key[1],
        "feature_set": best_binary_key[0],
        "features": best_binary_features,
        "selection_metric": "recall_heavy, then f1_heavy and balanced_accuracy",
        "metrics": {k: float(best_binary_row[k]) for k in
                    ["accuracy", "balanced_accuracy", "precision_heavy", "recall_heavy", "f1_heavy", "macro_f1"]}
    },
    "limitations": [
        "Short observation window; results are not annual or seasonal validation.",
        "Heavy-rainfall classification is not actual flood prediction.",
        "Class imbalance can make accuracy misleading; inspect per-class metrics."
    ]
}
META_PATH = os.path.join(MODEL_DIR, "classification_model_metadata.json")
with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved multiclass model:", MULTI_MODEL_PATH)
print("Saved heavy-rainfall model:", BINARY_MODEL_PATH)
print("Saved multiclass metrics:", MULTI_METRICS_PATH)
print("Saved binary metrics:", BINARY_METRICS_PATH)
print("Saved predictions:", MULTI_PRED_PATH, "and", BINARY_PRED_PATH)
print("Saved metadata:", META_PATH)
print("Saved figures in:", FIG_DIR)


## Step 12 — Module 6 summary

After running all cells, record the actual results below. Do not invent or manually improve the metrics.

### Multiclass intensity model
- Best classifier: fill from `best_multi_row`
- Best feature set: fill from `best_multi_row`
- Accuracy:
- Balanced accuracy:
- Macro F1:
- Which rainfall classes had the lowest recall?

### Heavy-or-above model
- Best classifier: fill from `best_binary_row`
- Best feature set: fill from `best_binary_row`
- Precision for heavy-or-above:
- Recall for heavy-or-above:
- F1 for heavy-or-above:

### Interpretation
Macro F1 gives equal importance to each class and is useful when classes are imbalanced. Heavy-class recall measures the fraction of actual heavy-or-above days that the model identifies. Higher recall can come with more false alarms, so precision must be reported alongside it.

### Limitations
The dataset covers only a short date window. The classification results are not validated across a full monsoon cycle or multiple years. The binary target is a rainfall-threshold indicator, not a flood-occurrence label. Actual flood prediction requires observed flood labels and additional hydrological/geographical variables.

**Next:** Module 7 — Model comparison, interpretation and final evaluation across regression and classification.
